In [12]:
import numpy as np
import pandas as pd

In [13]:
# --------------------
# Configuration
# --------------------
input_csv = "results/fdata/predictions_original.csv"
output_csv = "results/fdata/predictions.csv"

In [14]:
# --------------------
# Load CSV (force column as string)
# --------------------
df = pd.read_csv(input_csv)

In [15]:
df.dtypes

job_id                      int64
user_id                     int64
time_submit                 int64
gt_runtime                  int64
pred_runtime_user           int64
pred_runtime_heuristic    float64
pred_runtime_dt             int64
pred_runtime_rnp            int64
pred_runtime_knn          float64
pred_runtime_llm          float64
dtype: object

In [16]:
df[df.pred_runtime_heuristic > df.pred_runtime_user]

,job_id,user_id,time_submit,gt_runtime,pred_runtime_user,pred_runtime_heuristic,pred_runtime_dt,pred_runtime_rnp,pred_runtime_knn,pred_runtime_llm
49,8688862,1177,1894051,2247,3600,16759.893805,73,1316,381.759302,331.0
727,8689125,1177,1894146,1450,3600,16632.587719,19027,153137,400.523571,1450.0
759,8689127,1177,1894151,1457,3600,16500.565217,1450,4884,407.709070,1457.0
785,8689138,1177,1894156,1533,3600,16370.879310,1450,7736,405.505270,1533.0
802,8689128,1177,1894160,1457,3600,16244.059829,1457,3554,386.483411,1457.0
...,...,...,...,...,...,...,...,...,...,...
84842,8771777,1775,2432250,1018,10800,11992.250711,988,1260,1109.320924,1030.0
84846,8771786,1775,2432550,1032,10800,11997.299210,988,1295,1068.896939,1032.0
84849,8771785,1775,2432627,1018,10800,12002.618882,980,1302,1174.405787,1018.0
84852,8771787,1775,2432695,1022,10800,12007.676870,1028,1296,1091.055336,1022.0


In [17]:
# 1. Convert to integer with correct approximation (round half away from zero)
df["pred_runtime_heuristic"] = (
    df["pred_runtime_heuristic"]
    .round()
    .astype(int)
)

df["pred_runtime_dt"] = (
    df["pred_runtime_dt"]
    .round()
    .astype(int)
)

df["pred_runtime_rnp"] = (
    df["pred_runtime_rnp"]
    .round()
    .astype(int)
)

df["pred_runtime_knn"] = (
    df["pred_runtime_knn"]
    .round()
    .astype(int)
)

df["pred_runtime_llm"] = (
    df["pred_runtime_llm"]
    .round()
    .astype(int)
)

In [18]:
# 2. Cap heuristic prediction to user prediction
df["pred_runtime_heuristic"] = np.minimum(
    df["pred_runtime_heuristic"],
    df["pred_runtime_user"]
)

df["pred_runtime_dt"] = np.minimum(
    df["pred_runtime_dt"],
    df["pred_runtime_user"]
)

df["pred_runtime_rnp"] = np.minimum(
    df["pred_runtime_rnp"],
    df["pred_runtime_user"]
)

df["pred_runtime_knn"] = np.minimum(
    df["pred_runtime_knn"],
    df["pred_runtime_user"]
)

df["pred_runtime_llm"] = np.minimum(
    df["pred_runtime_llm"],
    df["pred_runtime_user"]
)

In [19]:
input_old = "results/fdata/predictions_s.csv"

df_old = pd.read_csv(input_old)

In [20]:
df['pred_runtime_llm'] = ((df['pred_runtime_llm'] + df_old['pred_runtime_llm'])/2).round().astype(int)

In [21]:
df

,job_id,user_id,time_submit,gt_runtime,pred_runtime_user,pred_runtime_heuristic,pred_runtime_dt,pred_runtime_rnp,pred_runtime_knn,pred_runtime_llm
0,8687223,1122,1894043,462,2400,141,424,229,462,572
1,8773168,299,1894044,716,3600,523,670,896,691,710
2,8688742,299,1894044,1063,3600,523,752,919,708,783
3,8773169,299,1894044,721,3600,523,719,914,696,737
4,8773159,299,1894044,686,3600,523,703,928,715,704
...,...,...,...,...,...,...,...,...,...,...
84872,8771078,1214,2432755,658,900,627,900,900,669,694
84873,8771098,1214,2432878,659,900,627,900,900,670,658
84874,8771099,1214,2432894,665,900,627,900,900,668,664
84875,8771789,1775,2433016,1046,10800,10800,1028,581,1207,2658


In [27]:
# --------------------
# Save corrected CSV
# --------------------
df.to_csv(output_csv, index=False)

print(f"Corrected file saved to: {output_csv}")

Corrected file saved to: results/fdata/predictions.csv
